# Preprocessing exploration: borders, color, hair and segmentation

Each block tries several alternatives on the same 5 images and ends with a **decision cell**. The values chosen there feed the next block
(borders + resize -> color -> hair -> segmentation). All the code lives in `preprocessing/`; this notebook only calls it and **saves nothing**.

There are no ground-truth masks, so every choice is made by visual inspection.

In [ ]:
%matplotlib inline
import sys
from dataclasses import replace
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))        # repo folder (this notebook lives in Binary/ or Multiclass/)

from preprocessing import (PreprocParams, default_hair, select_images,
                           pick_sample, load_sample,
                           show_borders_resize, explore_color, explore_hair,
                           explore_seg_channels, explore_seg_methods, explore_seg_inputs, explore_morphology,
                           stage_crop, stage_resize, stage_color, stage_hair,
                           print_params, final_summary)

# ---- adapt to your dataset ----
DATA_ROOT = Path("../dataset_binary")                       # folder with <split>/<class>/ images
CLASSES = ["nevus", "others"]                  # class folder names
SPLIT = "train"
SEED = 42

class_dir = lambda split, cls: DATA_ROOT / split / cls

## 0. Sample

3 random images spread across the classes + the 2 hairiest of a random pool of 60 other images (so that the hair step is actually exercised).

In [ ]:
jobs = select_images(class_dir, CLASSES, SPLIT, n=None, seed=SEED)
sample = pick_sample(jobs, CLASSES, n_random=3, n_hairy=2, seed=SEED, pool=60, split=SPLIT)
images = load_sample(sample)                      # {name: RGB}

## 1. Black borders and resize

Chosen version: black borders detected with a dark threshold, central square of the useful rectangle, resize to 512 px.

In [ ]:
P = PreprocParams(dark_thr=35, min_area_frac=0.08, target_size=512)
cropped = stage_crop(images, P)
resized = stage_resize(cropped, P)

In [ ]:
show_borders_resize(images, P)

## 2. Color constancy

No correction, Gray World, White Patch, Shades of Gray with p = 3 / 6 / 10 (light mapped to a gray level) and SoG p = 6 correcting only the tint.

In [ ]:
explore_color(resized, target=P.sog_target)

**Decision (color).**

In [ ]:
P = replace(P, sog_p=6, normalize_brightness=True, sog_target=195.0)
sog = stage_color(resized, P)

## 3. Hair removal

Cross vs lines kernel, kernel size, threshold, gray vs max-RGB channel, with / without the shape filter, and detection at 256 px vs 512 px.
One figure per image: hair mask on top, inpainted result below.

In [ ]:
explore_hair(sog)

**Decision (hair).**

In [ ]:
hair = default_hair()
hair.update(kernel_size=5, thresh=15, kernel_shape="cross", channel="gray", filter_shape=True)
P = replace(P, hair=hair, hair_ref_side=256)
clean = stage_hair(sog, P)

## 4. Segmentation

All the comparisons use the hair-free images from the previous decision.

**4a.** Channel for Otsu.

In [ ]:
explore_seg_channels(clean, channels=('L', 'a', 'b', 'Cr', 'S', 'gray'))

**4b.** Otsu vs K-means, GrabCut, Watershed and Chan-Vese (all followed by the same cleaning).

In [ ]:
explore_seg_methods(clean, channel='S')

**4c.** Input to the segmentation: no blur, median blur, or CLAHE (on L of Lab).

In [ ]:
explore_seg_inputs(clean, channel='S')

**4d.** Morphological cleaning: raw Otsu vs opening / closing of different sizes.

In [ ]:
explore_morphology(clean, channel='S')

**Decision (segmentation).** CLAHE (4c) gives no visible improvement over no blur, and it even worsens some masks, so it is not used. Final choice: Otsu(S) on the hair-free image, no blur.

In [ ]:
P = replace(P, seg_channel="S", open_k=5, close_k=15, center_sigma=0.35)

## 5. Final result

Parameters finally chosen and the full pipeline applied to the sample, one row per image.

In [ ]:
print_params(P)

In [ ]:
final_summary(sample, P)